# Day 1 — Train / Validation / Test Splits

Building a correct three-way split on the Breast Cancer Wisconsin dataset, tuning a hyperparameter against the validation set only, and evaluating the final model on the test set exactly once.

## Loading the Dataset
Load the dataset and check its shape and structure before splitting anything.

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [19]:
df = pd.read_csv("breast-cancer.csv")
print (df.shape)
df.info()

(569, 32)
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             56

The dataset has 569 rows and 32 columns: a patient id, the diagnosis label, and 30 numeric measurements — no missing values, so no cleaning is needed before splitting.

In [20]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


.head() gives a first look at the raw feature values before any encoding or scaling is applied.

In [21]:
df["diagnosis"] = df["diagnosis"].map({
     "B" : 0,
     "M" : 1

})

X = df.drop(["id", "diagnosis"], axis =1)
y = df["diagnosis"]

diagnosis is mapped to numeric (B → 0, M → 1), and the data is separated into features X (dropping id and diagnosis) and target y (diagnosis).

## Step 1 — Creating the 60/20/20 Train/Validation/Test Split
Carve off the test set first with one train_test_split call, then split the remaining data into training and validation with a second call — the exact two-step recipe from the lesson.

In [22]:
X_temp , X_test , y_temp , y_test = train_test_split(
   X ,
   y,
    test_size = 0.2,
    random_state = 42
    
)


X_train , X_val , y_train , y_val = train_test_split(
   X_temp,
    y_temp,
    test_size = 0.25,
    random_state = 42
    
)

The first split holds out 20% as X_test/y_test, untouched from here on. The second split takes the remaining 80% (X_temp/y_temp) and splits it 75/25, which works out to 60% train / 20% validation of the original data.

In [23]:
print ("traning" , X_train.shape , y_train.shape)
print ("validation" , X_val.shape , y_val.shape)
print ("testing" , X_test.shape , y_test.shape)

traning (341, 30) (341,)
validation (114, 30) (114,)
testing (114, 30) (114,)


In [24]:
scaler = StandardScaler()

scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

The scaler is fit only on X_train, then used to transform all three sets. This keeps both the validation and test sets honestly "unseen" by the scaler's fitting process.

## Step 2 — Training and Tuning Against the Validation Set
Train a first logistic regression model and check its accuracy on the validation set — never the test set — to see how well it's doing.

In [26]:
model = LogisticRegression()

model.fit(X_train_scaled , y_train)

y_val_pred = model.predict(X_val_scaled)

val_accuracy = accuracy_score(y_val, y_val_pred)

print("Validation Accuracy:", val_accuracy)

Validation Accuracy: 0.9736842105263158


**First validation accuracy ≈ 97.4%** with default settings — a strong starting point, and importantly, this number comes from the validation set, not the test set, so it's safe to look at repeatedly while tuning.

In [29]:
C_values = [0.01, 0.5, 1, 10, 100]
results = []

for c in C_values :
    model = LogisticRegression( C = c )
    model.fit(X_train_scaled , y_train)

    y_val_pred = model.predict(X_val_scaled)

    val_accuracy = accuracy_score(y_val, y_val_pred)

    print("C =", c, "Validation Accuracy:", val_accuracy)
    
    results.append(val_accuracy)

C = 0.01 Validation Accuracy: 0.956140350877193
C = 0.5 Validation Accuracy: 0.9736842105263158
C = 1 Validation Accuracy: 0.9736842105263158
C = 10 Validation Accuracy: 0.9649122807017544
C = 100 Validation Accuracy: 0.9649122807017544


**Tuning `C` (the regularization strength) against the validation set:**

| C | Validation Accuracy |
|---|---|
| 0.01 | 95.6% |
| 0.5 | **97.4%** |
| 1 (default) | **97.4%** |
| 10 | 96.5% |
| 100 | 96.5% |

C = 0.5 and C = 1 tie for the best validation accuracy. C = 0.5 is chosen for the final model — it applies slightly stronger regularization than the default while matching its performance, which tends to generalize at least as well and is a reasonable, well-justified tie-breaker. 

## Step 3 — Final, One-Time Evaluation on the Test Set
With C = 0.5 selected as the final choice, train the final model and evaluate it on the test set exactly once — no further tuning after this point.

In [30]:
final_model = LogisticRegression(C=0.5)

final_model.fit (X_train_scaled , y_train)

y_test_pred = final_model.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, y_test_pred)

print("Final Test Accuracy:", test_accuracy)

Final Test Accuracy: 0.9824561403508771


**Final Test Accuracy ≈ 98.2%.** This is now the honest, one-time estimate of how the model would perform on genuinely new data — it wasn't used to make any decisions about the model, unlike the validation set, which was checked five times during tuning.

## Step 4 — What Would Go Wrong Tuning Against the Test Set

If the 5 C values above had been checked against the test set instead of the validation set, the final reported test accuracy would no longer be trustworthy. Each time a choice is made by looking at how well it performs on a specific set of data, information about that data leaks into the decision — even though no code ever directly trains on it. By the time the "best" C was picked, it would have been picked *because* it happened to do well on those particular 114 test rows, not necessarily because it generalizes best overall.

